In [2]:
"""
=============================================================
  Baseline CNN — Static Hand-Sign Alphabet Classification
  Dataset : Sign Language MNIST
  Framework: TensorFlow / Keras
=============================================================
STEPS TO RUN
------------
1. Install dependencies
      pip install tensorflow pandas numpy matplotlib scikit-learn seaborn kaggle

2. Download the dataset (choose ONE method)
   a) Kaggle CLI (recommended)
         kaggle datasets download -d datamunge/sign-language-mnist
         unzip sign-language-mnist.zip -d data/
   b) Manual → https://www.kaggle.com/datasets/datamunge/sign-language-mnist
      Place sign_mnist_train.csv and sign_mnist_test.csv inside ./data/

3. Run the script
      python sign_language_cnn.py

4. Launch TensorBoard (in a separate terminal)
      tensorboard --logdir=logs/

5. Outputs saved to ./outputs/
      - training_history.png   (accuracy & loss curves)
      - confusion_matrix.png   (per-class performance)
      - best_model.keras       (best weights, saved by ModelCheckpoint)
      - sample_predictions.png (visual spot-check on test images)
=============================================================
"""

'\n=============================================================\n  Baseline CNN — Static Hand-Sign Alphabet Classification\n  Dataset : Sign Language MNIST\n  Framework: TensorFlow / Keras\n=============================================================\nSTEPS TO RUN\n------------\n1. Install dependencies\n      pip install tensorflow pandas numpy matplotlib scikit-learn seaborn kaggle\n\n2. Download the dataset (choose ONE method)\n   a) Kaggle CLI (recommended)\n         kaggle datasets download -d datamunge/sign-language-mnist\n         unzip sign-language-mnist.zip -d data/\n   b) Manual → https://www.kaggle.com/datasets/datamunge/sign-language-mnist\n      Place sign_mnist_train.csv and sign_mnist_test.csv inside ./data/\n\n3. Run the script\n      python sign_language_cnn.py\n\n4. Launch TensorBoard (in a separate terminal)\n      tensorboard --logdir=logs/\n\n5. Outputs saved to ./outputs/\n      - training_history.png   (accuracy & loss curves)\n      - confusion_matrix.png   (p

In [3]:
# ── 0. Imports ────────────────────────────────────────────
import os, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import classification_report, confusion_matrix

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, regularizers

print(f"TensorFlow  : {tf.__version__}")
print(f"GPU available: {bool(tf.config.list_physical_devices('GPU'))}\n")

TensorFlow  : 2.21.0
GPU available: False



In [4]:
# ── 1. Hyperparameters & Paths ────────────────────────────
TRAIN_CSV   = "Dataset/sign_mnist_train/sign_mnist_train.csv"
TEST_CSV    = "Dataset/sign_mnist_test/sign_mnist_test.csv"

OUTPUT_DIR  = "outputs"
LOG_DIR     = "logs"          # TensorBoard log directory
IMG_SIZE    = 28              # original pixel width/height
CHANNELS    = 1               # grayscale
NUM_CLASSES = 24              # A-Z minus J(9) and Z(25) — no motion signs
BATCH_SIZE  = 64
EPOCHS      = 30
LR          = 1e-3
SEED        = 42

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)
tf.random.set_seed(SEED)
np.random.seed(SEED)

In [5]:
# ── 2. Load & Pre-process Data ────────────────────────────
def load_data(path):
    """Read CSV → (images float32 [0,1], integer labels)."""
    df     = pd.read_csv(path)
    labels = df["label"].values.astype(np.int32)
    pixels = df.drop("label", axis=1).values.astype(np.float32)

    # Normalise to [0, 1]
    pixels /= 255.0

    # Reshape to (N, 28, 28, 1)
    images = pixels.reshape(-1, IMG_SIZE, IMG_SIZE, CHANNELS)
    return images, labels

print("Loading data …")
X_train, y_train = load_data(TRAIN_CSV)
X_test,  y_test  = load_data(TEST_CSV)

# Sign-Language MNIST skips J (9) and Z (25).
# Map raw labels to 0-based class indices
LABEL_MAP   = {v: i for i, v in enumerate(sorted(set(y_train)))}
CLASS_NAMES = [chr(ord("A") + k) for k in sorted(LABEL_MAP.keys())
               if chr(ord("A") + k) not in ("J", "Z")]

y_train_idx = np.array([LABEL_MAP[l] for l in y_train])
y_test_idx  = np.array([LABEL_MAP[l] for l in y_test])

print(f"Train : {X_train.shape}  |  Test : {X_test.shape}")
print(f"Classes ({NUM_CLASSES}): {CLASS_NAMES}\n")

Loading data …
Train : (27455, 28, 28, 1)  |  Test : (7172, 28, 28, 1)
Classes (24): ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y']



In [6]:
# ── 3. Baseline CNN Architecture ─────────────────────────
def build_baseline_cnn(input_shape, num_classes):
    """
    A clean 3-block CNN:
      Block 1 : 32 filters  → MaxPool → Dropout
      Block 2 : 64 filters  → MaxPool → Dropout
      Block 3 : 128 filters → MaxPool → Dropout
      Head    : Flatten → Dense(256) → Dense(num_classes, softmax)
    """
    inp = layers.Input(shape=input_shape, name="input")

    # ── Block 1 ──
    x = layers.Conv2D(32, (3, 3), padding="same", activation="relu",
                      kernel_regularizer=regularizers.l2(1e-4), name="conv1_1")(inp)
    x = layers.BatchNormalization(name="bn1_1")(x)
    x = layers.Conv2D(32, (3, 3), padding="same", activation="relu",
                      kernel_regularizer=regularizers.l2(1e-4), name="conv1_2")(x)
    x = layers.BatchNormalization(name="bn1_2")(x)
    x = layers.MaxPooling2D((2, 2), name="pool1")(x)
    x = layers.Dropout(0.25, name="drop1")(x)

    # ── Block 2 ──
    x = layers.Conv2D(64, (3, 3), padding="same", activation="relu",
                      kernel_regularizer=regularizers.l2(1e-4), name="conv2_1")(x)
    x = layers.BatchNormalization(name="bn2_1")(x)
    x = layers.Conv2D(64, (3, 3), padding="same", activation="relu",
                      kernel_regularizer=regularizers.l2(1e-4), name="conv2_2")(x)
    x = layers.BatchNormalization(name="bn2_2")(x)
    x = layers.MaxPooling2D((2, 2), name="pool2")(x)
    x = layers.Dropout(0.25, name="drop2")(x)

    # ── Block 3 ──
    x = layers.Conv2D(128, (3, 3), padding="same", activation="relu",
                      kernel_regularizer=regularizers.l2(1e-4), name="conv3_1")(x)
    x = layers.BatchNormalization(name="bn3_1")(x)
    x = layers.MaxPooling2D((2, 2), name="pool3")(x)
    x = layers.Dropout(0.25, name="drop3")(x)

    # ── Classification Head ──
    x = layers.Flatten(name="flatten")(x)
    x = layers.Dense(256, activation="relu",
                     kernel_regularizer=regularizers.l2(1e-4), name="fc1")(x)
    x = layers.BatchNormalization(name="bn_fc")(x)
    x = layers.Dropout(0.50, name="drop_fc")(x)
    out = layers.Dense(num_classes, activation="softmax", name="output")(x)

    model = models.Model(inp, out, name="Baseline_CNN")
    return model


model = build_baseline_cnn(
    input_shape=(IMG_SIZE, IMG_SIZE, CHANNELS),
    num_classes=NUM_CLASSES,
)
model.summary()

Model: "Baseline_CNN"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input (InputLayer)              │ (None, 28, 28, 1)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1_1 (Conv2D)                │ (None, 28, 28, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn1_1 (BatchNormalization)      │ (None, 28, 28, 32)     │           128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1_2 (Conv2D)                │ (None, 28, 28, 32)     │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn1_2 (BatchNormalization)      │ (None, 28, 28, 32)     │           128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool1 (MaxPooling2D)            │ (None, 14, 14, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ drop1 (Dropout)                 │ (None, 14, 14, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2_1 (Conv2D)                │ (None, 14, 14, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn2_1 (BatchNormalization)      │ (None, 14, 14, 64)     │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2_2 (Conv2D)                │ (None, 14, 14, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn2_2 (BatchNormalization)      │ (None, 14, 14, 64)     │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool2 (MaxPooling2D)            │ (None, 7, 7, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ drop2 (Dropout)                 │ (None, 7, 7, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv3_1 (Conv2D)                │ (None, 7, 7, 128)      │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn3_1 (BatchNormalization)      │ (None, 7, 7, 128)      │           512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool3 (MaxPooling2D)            │ (None, 3, 3, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ drop3 (Dropout)                 │ (None, 3, 3, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 1152)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ fc1 (Dense)                     │ (None, 256)            │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn_fc (BatchNormalization)      │ (None, 256)            │         1,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ drop_fc (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, 24)             │         6,168 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 442,488 (1.69 MB)

 Trainable params: 441,336 (1.68 MB)

 Non-trainable params: 1,152 (4.50 KB)

In [7]:
# ── 4. Compile ────────────────────────────────────────────
optimizer = tf.keras.optimizers.Adam(learning_rate=LR)

model.compile(
    optimizer=optimizer,
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

In [8]:
# ── 5. Callbacks ──────────────────────────────────────────
cb_list = [
    # Save the best weights based on val_accuracy
    callbacks.ModelCheckpoint(
        filepath=os.path.join(OUTPUT_DIR, "best_model.keras"),
        monitor="val_accuracy",
        save_best_only=True,
        verbose=1,
    ),
    # Halve LR when val_loss stops improving
    callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=4,
        min_lr=1e-6,
        verbose=1,
    ),
    # Stop early when val_accuracy plateaus
    callbacks.EarlyStopping(
        monitor="val_accuracy",
        patience=8,
        restore_best_weights=True,
        verbose=1,
    ),
    # TensorBoard — logs written to LOG_DIR every epoch
    # Launch with:  tensorboard --logdir=logs/
    callbacks.TensorBoard(
        log_dir=LOG_DIR,
        histogram_freq=1,      # log weight histograms each epoch
        write_graph=True,      # visualise the model graph
        write_images=True,     # log model weights as images
        update_freq="epoch",   # write scalars once per epoch
    ),
]

print(f"\nTensorBoard logs → {LOG_DIR}/")
print("Run in a separate terminal:  tensorboard --logdir=logs/\n")


TensorBoard logs → logs/
Run in a separate terminal:  tensorboard --logdir=logs/



In [9]:
# ── 6. Train ──────────────────────────────────────────────
print("── Training ─────────────────────────────────────")
history = model.fit(
    X_train, y_train_idx,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    validation_data=(X_test, y_test_idx),
    callbacks=cb_list,
    verbose=1,
)

── Training ─────────────────────────────────────
Epoch 1/30
429/429 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step - accuracy: 0.5100 - loss: 1.8805
Epoch 1: val_accuracy improved from None to 0.62730, saving model to outputs/best_model.keras
429/429 ━━━━━━━━━━━━━━━━━━━━ 25s 53ms/step - accuracy: 0.7396 - loss: 0.9592 - val_accuracy: 0.6273 - val_loss: 1.2564 - learning_rate: 0.0010
Epoch 2/30
429/429 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step - accuracy: 0.9692 - loss: 0.1863
Epoch 2: val_accuracy improved from 0.62730 to 0.95636, saving model to outputs/best_model.keras
429/429 ━━━━━━━━━━━━━━━━━━━━ 25s 58ms/step - accuracy: 0.9777 - loss: 0.1568 - val_accuracy: 0.9564 - val_loss: 0.2018 - learning_rate: 0.0010
Epoch 3/30
429/429 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step - accuracy: 0.9915 - loss: 0.1070
Epoch 3: val_accuracy improved from 0.95636 to 0.97309, saving model to outputs/best_model.keras
429/429 ━━━━━━━━━━━━━━━━━━━━ 26s 61ms/step - accuracy: 0.9926 - loss: 0.1007 - val_accuracy: 0.9731 - val_loss: 0.1

In [10]:
# ── 7. Evaluate ───────────────────────────────────────────
print("\n── Evaluation ───────────────────────────────────")
loss, acc = model.evaluate(X_test, y_test_idx, verbose=0)
print(f"Test Loss     : {loss:.4f}")
print(f"Test Accuracy : {acc * 100:.2f}%\n")

y_pred_prob = model.predict(X_test, verbose=0)
y_pred      = np.argmax(y_pred_prob, axis=1)

print(classification_report(
    y_test_idx, y_pred,
    target_names=CLASS_NAMES,
    digits=4,
))


── Evaluation ───────────────────────────────────
Test Loss     : 0.0472
Test Accuracy : 99.36%

              precision    recall  f1-score   support

           A     1.0000    1.0000    1.0000       331
           B     1.0000    1.0000    1.0000       432
           C     1.0000    1.0000    1.0000       310
           D     1.0000    1.0000    1.0000       245
           E     1.0000    1.0000    1.0000       498
           F     1.0000    1.0000    1.0000       247
           G     0.9457    1.0000    0.9721       348
           H     1.0000    0.9541    0.9765       436
           I     1.0000    1.0000    1.0000       288
           K     1.0000    1.0000    1.0000       331
           L     1.0000    1.0000    1.0000       209
           M     0.9517    1.0000    0.9752       394
           N     1.0000    0.9141    0.9551       291
           O     1.0000    1.0000    1.0000       246
           P     1.0000    1.0000    1.0000       347
           Q     1.0000    1.0000    

In [11]:
# ── 8. Plot: Training History ─────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Baseline CNN — Training History", fontsize=14, fontweight="bold")

axes[0].plot(history.history["accuracy"],     label="Train Acc")
axes[0].plot(history.history["val_accuracy"], label="Val Acc")
axes[0].set_title("Accuracy")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Accuracy")
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(history.history["loss"],     label="Train Loss")
axes[1].plot(history.history["val_loss"], label="Val Loss")
axes[1].set_title("Loss")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Loss")
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
path_hist = os.path.join(OUTPUT_DIR, "training_history.png")
plt.savefig(path_hist, dpi=150)
print(f"Saved → {path_hist}")
plt.close()

Saved → outputs/training_history.png


In [12]:
# ── 9. Plot: Confusion Matrix ─────────────────────────────
cm = confusion_matrix(y_test_idx, y_pred)
fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
    linewidths=0.4, ax=ax,
)
ax.set_title(f"Confusion Matrix  (Test Acc = {acc*100:.2f}%)",
             fontsize=13, fontweight="bold")
ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
plt.tight_layout()
path_cm = os.path.join(OUTPUT_DIR, "confusion_matrix.png")
plt.savefig(path_cm, dpi=150)
print(f"Saved → {path_cm}")
plt.close()

Saved → outputs/confusion_matrix.png


In [13]:
# ── 10. Plot: Sample Predictions ──────────────────────────
def plot_samples(images, true_labels, pred_labels, class_names, n=20):
    indices = np.random.choice(len(images), n, replace=False)
    cols = 5; rows = n // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 2.5, rows * 2.8))
    fig.suptitle("Sample Predictions  (Green = correct, Red = wrong)",
                 fontsize=12, fontweight="bold")
    for i, idx in enumerate(indices):
        ax = axes[i // cols][i % cols]
        ax.imshow(images[idx].squeeze(), cmap="gray")
        t = class_names[true_labels[idx]]
        p = class_names[pred_labels[idx]]
        color = "green" if t == p else "red"
        ax.set_title(f"T:{t}  P:{p}", color=color, fontsize=9)
        ax.axis("off")
    plt.tight_layout()
    return fig


fig_sp = plot_samples(X_test, y_test_idx, y_pred, CLASS_NAMES)
path_sp = os.path.join(OUTPUT_DIR, "sample_predictions.png")
fig_sp.savefig(path_sp, dpi=150)
print(f"Saved → {path_sp}")
plt.close()

print("\n✅  All done!  Outputs are in ./outputs/")
print("📊  Open TensorBoard:  tensorboard --logdir=logs/")

Saved → outputs/sample_predictions.png

✅  All done!  Outputs are in ./outputs/
📊  Open TensorBoard:  tensorboard --logdir=logs/
